## Data Cleaning for ML

### Step 1 — Load the ML Dataset

In [0]:
ml_df = spark.table('retail_project.gold.ml_sales_dataset')

In [0]:
display(ml_df)

In [0]:
#Checking the Rows and columns of the dataframe
print("Rows :", ml_df.count())
print("Columns :", len(ml_df.columns))

#### Step 2 — Verify Schema Again

In [0]:
ml_df.printSchema()

Why?

- Sometimes feature engineering changes data types. Always verify before modeling.

#### Step 3 — Missing Values After Filtering

In [0]:
from pyspark.sql.functions import col, count, when

null_df = ml_df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in ml_df.columns
])

display(null_df)

Goal

After removing rows with missing Total_revenue, we want to know:

- Which columns still have missing values?
- Are they acceptable?
- Do we need imputation?

#### Step 3.1: Calculate Missing Value Percentage

In [0]:
from pyspark.sql.functions import col, count, when, round

total_rows = ml_df.count()

null_df_percentage = ml_df.select([
    round(
        (count(when(col(c).isNull(), c)) / total_rows) * 100, 2
    ).alias(c)
    for c in ml_df.columns
])

display(null_df_percentage)

### **Inference from Missing Value Percentage**

* **`loyalty_points`** and **`Avg_loyalty_points`** have the highest missing values (**71.32%**), indicating that most customers are not enrolled in the loyalty program or loyalty information is unavailable.
* **Customer-related columns** (`customer_name`, `gender`, `city`, and `state`) each have **13.10%** missing values, suggesting incomplete customer profile information.
* **`product_id`** and **`customer_id`** have only **1.85%** missing values, indicating that key identifiers are largely complete.
* **No missing values** are present in `product_name`, `category`, `Year`, `Month`, `order_date`, `Total_revenue`, `Total_quantity`, `Total_orders`, `Avg_revenue`, `Avg_quantity`, `Min_price`, and `Max_price`, showing that the core transaction and revenue data is complete and reliable.
* For machine learning, prioritize handling missing values in **`loyalty_points`**, **`gender`**, **`city`**, and **`state`**, while identifier columns such as **`customer_name`** and **`customer_id`** can generally be dropped if they are not required as predictive features.


| Column             | Missing Values | Recommendation                                                                         |
| ------------------ | -------------: | -------------------------------------------------------------------------------------- |
| product_id         |             70 | Keep (important feature). Impute or drop only those 70 rows if needed.                 |
| customer_id        |             70 | Drop for ML (identifier, not predictive).                                              |
| customer_name      |            496 | Drop (identifier, not useful for prediction).                                          |
| gender             |            496 | Keep and impute with **"Unknown"** or the mode.                                        |
| city               |            496 | Keep and impute with **"Unknown"**.                                                    |
| state              |            496 | Keep and impute with **"Unknown"**.                                                    |
| loyalty_points     |           2701 | Keep. Impute with **0** (if NULL means not enrolled) or use the median if appropriate. |
| Avg_loyalty_points |           2701 | Drop to avoid redundancy if `loyalty_points` is retained.                              |


These Columns to Drop for ML

These columns are identifiers and generally do not help the model learn patterns:

- ❌ customer_id
- ❌ customer_name
- ❌ Avg_loyalty_points (if it is derived from loyalty_points and adds redundant information).

Theses columns droped because of not important for predictive purpose.

In [0]:
ml_df_cleaned = ml_df.drop('customer_name', 'customer_id', 'Avg_loyalty_points')

display(ml_df_cleaned)

In [0]:
#Checking the Rows and columns of the dataframe
print("Rows :", ml_df_cleaned.count())
print("Columns :", len(ml_df_cleaned.columns))

#### Treating Missing Values:

- `gender`, `city`, and `state` - have some missing values, which can be imputed with "Unknown" to preserve records and improve data consistency.

In [0]:
ml_df_cleaned = ml_df_cleaned.fillna({
    'gender': 'U',
    'city' : 'Unknown',
    'state' : 'Unknown'
})

display(ml_df_cleaned)

In [0]:
from pyspark.sql.functions import col, count, when

check_null_df = ml_df_cleaned.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["gender", "city", "state"]
]
)

display(check_null_df)

In [0]:
#Treating null values in loyalty_points using median
from pyspark.sql.functions import col, when

# Calculate median
median_loyalty = ml_df_cleaned.approxQuantile("loyalty_points", [0.5], 0.01)[0]

print("Median Loyalty Points:", median_loyalty)

# Impute null values with median
ml_df_cleaned = ml_df_cleaned.withColumn(
    "loyalty_points",
    when(col("loyalty_points").isNull(), median_loyalty)
    .otherwise(col("loyalty_points"))
)

display(ml_df_cleaned)

In [0]:
from pyspark.sql.functions import col, count, when

null_df = ml_df_cleaned.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in ml_df_cleaned.columns
])

display(null_df)

### **Inference After Missing Value Treatment**

* ✅ All missing values have been successfully handled **except** for **`product_id`**.
* **`product_id`** still contains **70 missing values**, which is only a small percentage of the dataset.
* Since `product_id` is an **identifier** and not a predictive feature, it has limited importance for machine learning.
* If `product_name` uniquely identifies the product, the missing `product_id` values **can be ignored or the column can be dropped** before model training.
* The remaining dataset is **clean and ready for feature engineering and machine learning**, with no missing values in the target variable (`Total_revenue`) or other important feature columns.

### **Recommendation**

* **For EDA:** Keep `product_id` as it helps identify products.
* **For ML:** Drop `product_id` if `product_name` (or an encoded version of it) is used as a feature, since `product_id` is only an identifier and does not provide predictive information.


Since **`product_id`** is an important feature for EDA and only a very small portion of the dataset contains missing values (**70 records, 1.85% of the dataset**), it is reasonable to drop these records. This has a minimal impact on the overall dataset while ensuring better data quality for further analysis.

In [0]:
from pyspark.sql.functions import col

ml_df_cleaned = ml_df_cleaned.filter(col("product_id").isNotNull())
display(ml_df_cleaned)

In [0]:
from pyspark.sql.functions import col, count, when

null_df = ml_df_cleaned.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in ml_df_cleaned.columns
])

display(null_df)

- All Missing Values are treated/imputed succussfully.

In [0]:
#Checking the Rows and columns of the dataframe
print("Rows :", ml_df_cleaned.count())
print("Columns :", len(ml_df_cleaned.columns))

#### Step 4 — Summary Statistics

In [0]:
ml_df.describe().display()

In [0]:
ml_df_cleaned.describe().display()

### **Inference from Summary Statistics (After Data Cleaning)**

* The dataset contains **3,717 complete records** after handling all missing values.
* The data covers **2025**, with transactions recorded across **3 months (January–March)**.
* The **average revenue** is **75,920.09**, with values ranging from **1,015.00** to **239,685.00**, indicating a wide variation in sales.
* The **standard deviation of revenue (55,745.22)** is high, suggesting significant variability in transaction values.
* Customers purchase an average of **1.99 products per order**, with a maximum of **5 products**, indicating that most orders contain only a few items.
* The average number of orders is **1.00**, showing that most records correspond to a single order.
* The average **loyalty points** is **1,051.86**, ranging from **118** to **2,000**, indicating different levels of customer engagement.
* Product prices range from **1,007.00** to **79,959.00**, showing that the dataset includes both low-priced and premium products.
* Categorical columns show products from multiple categories, cities, states, and customer genders, providing good diversity for further analysis.

### **Key Insights**

* ✅ The dataset is **clean with no missing values**.
* ✅ Revenue and product prices have **high variability**, making them suitable for further EDA and predictive modeling.
* ✅ Most transactions involve **1–2 products per order**, which reflects typical retail purchasing behavior.
* ✅ The dataset is now **ready for feature engineering, visualization, and machine learning model development**.


**Note:** The `loyalty_points` feature is not expected to contribute significantly to predicting `Total_revenue`. Therefore, it is better to drop this column before training the machine learning model to simplify the feature set and reduce unnecessary information.


In [0]:
ml_df_cleaned = ml_df_cleaned.drop('loyalty_points')
display(ml_df_cleaned)

In [0]:
#Checking the Rows and columns of the dataframe
print('Rows after cleaning :', ml_df_cleaned.count())
print('Columns after cleaning :', len(ml_df_cleaned.columns))

####Step 5 — Save the Cleaned ML Sales Dataset


In [0]:
ml_df_cleaned.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.ml_cleaned_sales_data')